# 🪨 LithoLens V2 — MineralImage5K Training
### 98 Minerals + Rock Gate | Complete Pipeline

**Before running:** Runtime → Change runtime type → **T4 GPU**

Run cells **one by one in order**. Total time: ~40–50 minutes.

## CELL 1 — Install Packages

In [ ]:
!pip install datasets timm onnx onnxruntime torch torchvision Pillow --quiet
print('✅ All packages installed')

## CELL 2 — Load MineralImage5K-98 from HuggingFace
No API key needed — it's public!

In [ ]:
from datasets import load_dataset
import os

print('⬇️  Downloading MineralImage5K-98 from HuggingFace...')
print('   This is ~3.9 GB — takes 5-10 minutes on Colab.')
dataset = load_dataset('Nech-C/mineralimage5K-98')

print(f'✅ Dataset loaded!')
print(f'   Train: {len(dataset["train"])} images')
print(f'   Val:   {len(dataset["validation"])} images')
print(f'   Test:  {len(dataset["test"])} images')

class_names = dataset['train'].features['name'].names
print(f'   Classes: {len(class_names)}')

## CELL 3 — Explore Classes + Add not_mineral Gate

In [ ]:
import numpy as np
from collections import Counter
import torchvision
from PIL import Image
import random

# Fix unusual class names
NAME_FIXES = {
    'labrador': 'labradorite',
    'nephritis': 'nephrite',
    'cancrinit': 'cancrinite',
    'scheelit': 'scheelite',
    'analcim': 'analcime',
    'elbait': 'elbaite',
}

cleaned_names = [NAME_FIXES.get(n, n) for n in class_names]

# Show sample counts
train_labels = dataset['train']['name']
label_counts = Counter(train_labels)
print('Samples per class:')
for idx, name in enumerate(class_names):
    count = label_counts.get(idx, 0)
    bar = '█' * (count // 10)
    print(f'  {cleaned_names[idx]:20s} {count:4d} {bar}')

# Download CIFAR-100 for not_mineral images
print('\n⬇️  Downloading CIFAR-100 for not_mineral class...')
cifar = torchvision.datasets.CIFAR100(root='/content/cifar100', train=True, download=True)

random.seed(42)
not_mineral_dir = '/content/not_mineral_images'
os.makedirs(not_mineral_dir, exist_ok=True)

indices = random.sample(range(len(cifar)), 800)
for i, idx in enumerate(indices):
    img, _ = cifar[idx]
    img = img.resize((224, 224), Image.BICUBIC)
    img.save(f'{not_mineral_dir}/not_mineral_{i:04d}.png')

print(f'✅ Created 800 not_mineral images')

# Final class list
FINAL_CLASS_NAMES = cleaned_names + ['not_mineral']
NUM_CLASSES = len(FINAL_CLASS_NAMES)
NUM_MINERAL_CLASSES = len(class_names)
NOT_MINERAL_IDX = NUM_CLASSES - 1
print(f'\n✅ Total classes: {NUM_CLASSES} ({NUM_MINERAL_CLASSES} minerals + not_mineral)')

## CELL 4 — Create DataLoaders

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from PIL import Image
import os

IMG_SIZE = 224
BATCH_SIZE = 32

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE + 32, IMG_SIZE + 32)),
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.RandomRotation(30),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])


class MineralDataset(Dataset):
    def __init__(self, hf_split, not_mineral_dir, transform, is_train=True):
        self.hf_data = hf_split
        self.transform = transform
        self.not_mineral_images = []
        if not_mineral_dir and os.path.exists(not_mineral_dir):
            all_imgs = sorted([
                os.path.join(not_mineral_dir, f)
                for f in os.listdir(not_mineral_dir) if f.endswith('.png')
            ])
            self.not_mineral_images = all_imgs[:640] if is_train else all_imgs[640:]
        self.total_len = len(self.hf_data) + len(self.not_mineral_images)

    def __len__(self):
        return self.total_len

    def __getitem__(self, idx):
        if idx < len(self.hf_data):
            item = self.hf_data[idx]
            image = item['image'].convert('RGB')
            label = item['name']
        else:
            nm_idx = idx - len(self.hf_data)
            image = Image.open(self.not_mineral_images[nm_idx]).convert('RGB')
            label = NOT_MINERAL_IDX
        if self.transform:
            image = self.transform(image)
        return image, label


train_dataset = MineralDataset(dataset['train'], not_mineral_dir, train_transform, True)
val_dataset = MineralDataset(dataset['validation'], not_mineral_dir, val_transform, False)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f'✅ Train: {len(train_dataset)} samples')
print(f'✅ Val:   {len(val_dataset)} samples')
print(f'✅ Batches/epoch: {len(train_loader)}')

## CELL 5 — Build Model + Class Weights

In [ ]:
import timm
import torch.nn as nn

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using: {device}')

model = timm.create_model('efficientnet_b0', pretrained=True, num_classes=NUM_CLASSES)
model = model.to(device)

# Class weights for imbalanced data
counts = [0] * NUM_CLASSES
for idx in range(NUM_MINERAL_CLASSES):
    counts[idx] = label_counts.get(idx, 1)
counts[NOT_MINERAL_IDX] = len(train_dataset.not_mineral_images)

w = 1.0 / (torch.tensor(counts, dtype=torch.float32) + 1)
w = w / w.sum() * NUM_CLASSES
class_weights = w.to(device)

print(f'✅ EfficientNet-B0 loaded — {sum(p.numel() for p in model.parameters()):,} params')

## CELL 6 — TRAIN (20 epochs, ~30-40 min on T4)
**Let this run. Don't interrupt it.**

In [ ]:
import torch.optim as optim
import time

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)

EPOCHS = 20
best_val_acc = 0.0
best_model_path = '/content/best_model.pth'
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

print('🚀 Training...')
print('=' * 65)

for epoch in range(EPOCHS):
    start = time.time()
    model.train()
    tl, tc, tt = 0.0, 0, 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        tl += loss.item()
        _, p = out.max(1)
        tt += y.size(0)
        tc += p.eq(y).sum().item()

    model.eval()
    vl, vc, vt = 0.0, 0, 0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            loss = criterion(out, y)
            vl += loss.item()
            _, p = out.max(1)
            vt += y.size(0)
            vc += p.eq(y).sum().item()

    scheduler.step()
    ta = 100.*tc/tt
    va = 100.*vc/vt
    history['train_loss'].append(tl/len(train_loader))
    history['train_acc'].append(ta)
    history['val_loss'].append(vl/len(val_loader))
    history['val_acc'].append(va)

    s = ''
    if va > best_val_acc:
        best_val_acc = va
        torch.save(model.state_dict(), best_model_path)
        s = '💾 SAVED'

    print(f'Epoch {epoch+1:2d}/{EPOCHS} | Train: {ta:.1f}% | Val: {va:.1f}% | {time.time()-start:.0f}s {s}')

print('=' * 65)
print(f'✅ Best validation accuracy: {best_val_acc:.1f}%')

## CELL 7 — Plot Training Results

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(history['train_acc'], label='Train', color='#2196F3')
ax1.plot(history['val_acc'], label='Val', color='#4CAF50')
ax1.set_title('Accuracy'); ax1.set_xlabel('Epoch'); ax1.set_ylabel('%')
ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(history['train_loss'], label='Train', color='#F44336')
ax2.plot(history['val_loss'], label='Val', color='#FF9800')
ax2.set_title('Loss'); ax2.set_xlabel('Epoch')
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.suptitle(f'LithoLens V2 — {NUM_CLASSES} classes — Best: {best_val_acc:.1f}%')
plt.tight_layout()
plt.savefig('/content/training_results.png', dpi=150, bbox_inches='tight')
plt.show()

## CELL 8 — Export to ONNX

In [ ]:
import json

model.load_state_dict(torch.load(best_model_path))
model.eval()
model.cpu()

dummy = torch.randn(1, 3, 224, 224)
onnx_path = '/content/litholens_model.onnx'

torch.onnx.export(
    model, dummy, onnx_path,
    export_params=True, opset_version=11,
    do_constant_folding=True,
    input_names=['input'], output_names=['output'],
    dynamic_axes={'input': {0: 'batch'}, 'output': {0: 'batch'}}
)

with open('/content/class_names.json', 'w') as f:
    json.dump({
        'classes': FINAL_CLASS_NAMES,
        'num_classes': NUM_CLASSES,
        'not_mineral_index': NOT_MINERAL_IDX
    }, f, indent=2)

sz = os.path.getsize(onnx_path) / (1024*1024)
print(f'✅ ONNX: {sz:.1f} MB | {NUM_CLASSES} classes')
print(f'   Input: "input" | Output: "output"')

## CELL 9 — Verify ONNX Works

In [ ]:
import onnxruntime as ort
import numpy as np

session = ort.InferenceSession('/content/litholens_model.onnx')
test = np.random.randn(1, 3, 224, 224).astype(np.float32)
out = session.run(None, {'input': test})[0][0]

sm = np.exp(out - out.max()) / np.sum(np.exp(out - out.max()))
top5 = np.argsort(sm)[::-1][:5]

print('✅ ONNX verified! Top 5 (random input):')
for i, idx in enumerate(top5):
    print(f'  {i+1}. {FINAL_CLASS_NAMES[idx]}: {sm[idx]*100:.1f}%')

## CELL 10 — Generate CSV Template for Geology Team
Give this CSV to your teammates. They fill in the geology data.
Only `label` and `name_english` are pre-filled.

In [ ]:
import csv

csv_path = '/content/minerals_template.csv'
columns = [
    'label', 'name_english', 'name_arabic', 'category',
    'hardness_moh', 'hardness_testable', 'luster', 'streak_color',
    'cleavage', 'fracture', 'acid_reaction', 'special_property',
    'description_for_ai', 'common_locations', 'primary_color'
]

with open(csv_path, 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(columns)
    for name in FINAL_CLASS_NAMES:
        if name == 'not_mineral':
            continue
        english = name.replace('_', ' ').title()
        if name == 'lapis_lazuli': english = 'Lapis Lazuli'
        row = [name, english] + [''] * (len(columns) - 2)
        writer.writerow(row)

print(f'✅ CSV template: {csv_path}')
print(f'   {NUM_MINERAL_CLASSES} minerals')
print(f'   Columns: {columns}')
print(f'')
print(f'📋 Give this to your geology teammates to fill in!')

## CELL 11 — Convert Filled CSV → minerals_db.json
⚠️ Run this AFTER your team fills the CSV.
Upload the filled CSV, then run this cell.

In [ ]:
import csv, json

# Uncomment these 2 lines when your team has filled the CSV:
# from google.colab import files
# uploaded = files.upload()

csv_file = '/content/minerals_template.csv'  # or use uploaded filename

minerals_db = {}
with open(csv_file, 'r', encoding='utf-8') as f:
    for row in csv.DictReader(f):
        minerals_db[row['label']] = {
            'name_english': row.get('name_english', ''),
            'name_arabic': row.get('name_arabic', ''),
            'category': row.get('category', ''),
            'hardness_moh': row.get('hardness_moh', ''),
            'hardness_testable': row.get('hardness_testable', ''),
            'luster': row.get('luster', ''),
            'streak_color': row.get('streak_color', ''),
            'cleavage': row.get('cleavage', ''),
            'fracture': row.get('fracture', ''),
            'acid_reaction': row.get('acid_reaction', ''),
            'special_property': row.get('special_property', ''),
            'description_for_ai': row.get('description_for_ai', ''),
            'common_locations': row.get('common_locations', ''),
            'primary_color': row.get('primary_color', ''),
        }

with open('/content/minerals_db.json', 'w', encoding='utf-8') as f:
    json.dump(minerals_db, f, indent=2, ensure_ascii=False)

print(f'✅ minerals_db.json — {len(minerals_db)} minerals')
print(f'   Place in: public/model/minerals_db.json')

## CELL 12 — Download Everything

In [ ]:
from google.colab import files

print('📥 Downloading files for the React app:')
print('  1. litholens_model.onnx  — AI model')
print('  2. class_names.json      — labels + not_mineral index')
print('  3. minerals_template.csv — for geology team')
print('  4. training_results.png  — for pitch deck')
print()

files.download('/content/litholens_model.onnx')
files.download('/content/class_names.json')
files.download('/content/minerals_template.csv')
files.download('/content/training_results.png')

print()
print('✅ Place in your React app:')
print('  public/model/litholens_model.onnx')
print('  public/model/class_names.json')
print('  public/model/minerals_db.json  (after team fills CSV)')